# AFSP full run

Operational runbook for the one long thread that produces the thesis's val numbers.
It runs on a **fresh A100** session and is organised into **four phases** because the
COMET stack and the generation stack cannot share one Python environment
(`requirements-comet.txt` pins `transformers==4.57.6` / `numpy==1.26.4`, the
generation stack pins `transformers==5.12.1` / `numpy==2.4.1`). The pipeline's own
ordering forces the alternation:

| Phase | Runtime | What runs | Depends on |
|------|---------|-----------|------------|
| 1 | **generation** (`requirements.txt`) | `build_index`, `afsp_sweep` (the ~26k-gen run) | — |
| 2 | **COMET** (`requirements-comet.txt`) | `afsp_verify` (COMET + judge), **freeze** | Phase 1 sweep result |
| 3 | **generation** (`requirements.txt`) | five-condition ladder generation + chrF/BLEU + stylometrics + judge Φ | frozen `(k, λ)` |
| 4 | **COMET** (`requirements-comet.txt`) | ladder COMET + paired bootstrap | Phase 3 ladder outputs |


---
## Phase 1 — generation runtime · the sweep

Fresh A100, default (generation) stack. This is the multi-hour run that gates
everything downstream.

In [8]:
# Confirm the GPU: the sweep regenerates with Qwen2.5-7B in bf16 (~15 GB weights).
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB


In [16]:
import os
if not os.path.isdir('Style-Aware-MT'):
    !git clone --branch feat/afsp-implementation https://github.com/prnamhr/Style-Aware-MT.git
%cd Style-Aware-MT
!git rev-parse --short HEAD

Cloning into 'Style-Aware-MT'...
remote: Enumerating objects: 479, done.
remote: Counting objects: 100% (108/108), done.
remote: Compressing objects: 100% (65/65), done.
remote: Total 479 (delta 54), reused 70 (delta 42), pack-reused 371 (from 1)
Receiving objects: 100% (479/479), 8.41 MiB | 22.49 MiB/s, done.
Resolving deltas: 100% (265/265), done.
/content/Style-Aware-MT/Style-Aware-MT/Style-Aware-MT/Style-Aware-MT
50c7477


In [26]:
# Generation stack.
!pip install -q -r requirements.txt

!pip uninstall -y torchvision torchaudio

The register centroid (`results/stylometrics_centroid.json`) is committed, so it is
already present. The kNN index is git-ignored and must be rebuilt each session.

In [5]:
import torch; print(torch.__version__, torch.cuda.is_available(), torch.version.cuda)

2.12.0+cu130 True 13.0


In [25]:
!python manage.py build_index --config configs/base_qwen.yaml

Embedding 10860 Persian/Arabic training sources with intfloat/multilingual-e5-large-instruct ...
Loading intfloat/multilingual-e5-large-instruct on device: cuda
Loading weights: 100% 391/391 [00:00<00:00, 4922.91it/s]
Batches: 100% 340/340 [00:07<00:00, 44.99it/s]
Wrote index to data/knn_index/ : embeddings (10860, 1024), 10860 pairs


In [27]:

!python manage.py afsp_sweep --config configs/afsp_sweep.yaml

Loading weights: 100% 339/339 [00:03<00:00, 87.84it/s]
skip afsp_k1_l0: outputs/sweep/afsp_k1_l0_val.jsonl exists (use --overwrite to regenerate)
skip afsp_k1_l0.25: outputs/sweep/afsp_k1_l0.25_val.jsonl exists (use --overwrite to regenerate)
skip afsp_k1_l0.5: outputs/sweep/afsp_k1_l0.5_val.jsonl exists (use --overwrite to regenerate)
skip afsp_k1_l0.75: outputs/sweep/afsp_k1_l0.75_val.jsonl exists (use --overwrite to regenerate)
skip afsp_k1_l1: outputs/sweep/afsp_k1_l1_val.jsonl exists (use --overwrite to regenerate)
skip afsp_k2_l0: outputs/sweep/afsp_k2_l0_val.jsonl exists (use --overwrite to regenerate)
skip afsp_k2_l0.25: outputs/sweep/afsp_k2_l0.25_val.jsonl exists (use --overwrite to regenerate)
skip afsp_k2_l0.5: outputs/sweep/afsp_k2_l0.5_val.jsonl exists (use --overwrite to regenerate)
skip afsp_k2_l0.75: outputs/sweep/afsp_k2_l0.75_val.jsonl exists (use --overwrite to regenerate)
skip afsp_k2_l1: outputs/sweep/afsp_k2_l1_val.jsonl exists (use --overwrite to regenerate)
ski

In [28]:
!git pull origin feat/afsp-implementation


From https://github.com/prnamhr/Style-Aware-MT
 * branch            feat/afsp-implementation -> FETCH_HEAD
Already up to date.


In [24]:
# The proxy-picked (k, lambda) the sweep recommends (free/local proxies only).
import json
sweep = json.load(open('results/afsp_sweep_val.json'))
rec = sweep.get('recommended')
print('proxy recommended:', rec and {k: rec[k] for k in ('tag', 'k', 'lambda', 'chrF', 'stylo_dist') if k in rec})

proxy recommended: {'tag': 'afsp_k16_l0.75', 'k': 16, 'lambda': 0.75, 'chrF': 40.05, 'stylo_dist': 0.3643}


---
## Phase 2 — COMET runtime · verify + freeze


In [30]:
%cd /content/Style-Aware-MT
# COMET stack (unbabel-comet + transformers 4.57.6 + numpy 1.26.4). Leaves openai in place
# from Phase 1's install, so the judge pass inside afsp_verify still works.
!pip install -q -r requirements-comet.txt

/content/Style-Aware-MT


In [31]:
# The judge is gpt-4.1 (configs/judge_eval.yaml) -> needs an OpenAI key.
import os, getpass
if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OPENAI_API_KEY: ')

OPENAI_API_KEY: ··········


In [33]:
!USE_TF=0 python -m src.infer.afsp_verify --config configs/afsp_sweep.yaml \
    --judge-config configs/judge_eval.yaml --top 3


Confirming top 3 proxy cells from afsp_sweep_val.json (['afsp_k16_l0.75', 'afsp_k8_l0.75', 'afsp_k8_l0.5']) on val
Loading checkpoint shards: 100% 4/4 [00:04<00:00,  1.06s/it]
skip afsp_k16_l0.75: outputs/sweep/afsp_k16_l0.75_val.jsonl exists (use --overwrite to regenerate)
skip afsp_k8_l0.75: outputs/sweep/afsp_k8_l0.75_val.jsonl exists (use --overwrite to regenerate)
skip afsp_k8_l0.5: outputs/sweep/afsp_k8_l0.5_val.jsonl exists (use --overwrite to regenerate)
Fetching 5 files:   0% 0/5 [00:00<?, ?it/s]
README.md: 3.40kB [00:00, 14.8MB/s]

checkpoints/model.ckpt:   0% 0.00/2.32G [00:00<?, ?B/s]

.gitattributes: 1.48kB [00:00, 10.6MB/s]
Fetching 5 files:  20% 1/5 [00:00<00:00,  6.86it/s]

hparams.yaml: 100% 567/567 [00:00<00:00, 6.55MB/s]


LICENSE: 9.69kB [00:00, 41.1MB/s]

checkpoints/model.ckpt:   0% 194k/2.32G [00:01<3:24:16, 190kB/s]
checkpoints/model.ckpt:   6% 134M/2.32G [00:01<00:19, 110MB/s]  
checkpoints/model.ckpt:  12% 268M/2.32G [00:02<00:12, 169MB/s]
checkpoints/model.ck

In [ ]:
# The freeze decision from the reported metrics.
import json
v = json.load(open('results/afsp_verify_val.json'))
print('freeze tag :', v['freeze'], '(proxy pick held)' if v['proxy_pick_held'] else '(runner-up overtook the proxy pick)')
for c in v['cells']:
    mark = '  <== freeze' if c['tag'] == v['freeze'] else ''
    print(f"  {c['tag']:<16} k={c['k']} lambda={c['lambda']}  COMET {c['comet_system']:.4f}  Phi {c['judge_mean']}{mark}")

### Freeze `(k, λ)` into `configs/base_qwen.yaml`


In [ ]:
import json, re, pathlib
v = json.load(open('results/afsp_verify_val.json'))
k = lam = None
for c in v['cells']:
    if c['tag'] == v['freeze']:
        k, lam = c['k'], c['lambda']
assert k is not None, 'freeze cell not found in results/afsp_verify_val.json'

p = pathlib.Path('configs/base_qwen.yaml')
text = p.read_text(encoding='utf-8')
text = re.sub(r'(?m)^(\s*k:\s*)\S+', lambda m: f'{m.group(1)}{k}', text, count=1)
text = re.sub(r'(?m)^(\s*lambda_style:\s*)\S+', lambda m: f'{m.group(1)}{lam}', text, count=1)
p.write_text(text, encoding='utf-8')
print(f'froze retrieval.k={k}, afsp.lambda_style={lam} into configs/base_qwen.yaml')
!grep -nE '^\s*(k|lambda_style):' configs/base_qwen.yaml

---
## Phase 3 — generation runtime · the five-condition ladder


In [ ]:
%cd /content/Style-Aware-MT
!pip install -q -r requirements.txt
# Same torchvision/torchaudio ABI mismatch as Phase 1 — drop them (text-only pipeline).
!pip uninstall -y torchvision torchaudio

In [ ]:
# Generate all five rungs on full val (each is resumable via its own outputs/*_val.jsonl).
CONDS = ['zeroshot', 'random_fewshot', 'knn_fewshot', 'afsp_margin', 'afsp_full']
for c in CONDS:
    print(f'\n=== generating {c} ===')
    !python manage.py infer --condition {c} --config configs/base_qwen.yaml

In [ ]:
# Adequacy proxies (chrF/BLEU) and register stylometrics — free/local, no COMET.
CONDS = 'zeroshot random_fewshot knn_fewshot afsp_margin afsp_full'
!python manage.py eval         --conditions {CONDS} --split val
!python manage.py stylometrics --conditions {CONDS} --split val

In [ ]:
# Register fidelity (judge Phi) over the ladder — OpenAI gpt-4.1.
import os, getpass
if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OPENAI_API_KEY: ')
CONDS = 'zeroshot random_fewshot knn_fewshot afsp_margin afsp_full'
!python manage.py judge --conditions {CONDS} --split val --config configs/judge_eval.yaml

---
## Phase 4 — COMET runtime · ladder COMET + paired bootstrap


In [ ]:
%cd /content/Style-Aware-MT
!pip install -q -r requirements-comet.txt

In [ ]:
CONDS = 'zeroshot random_fewshot knn_fewshot afsp_margin afsp_full'
!python manage.py comet --conditions {CONDS} --split val

In [ ]:
CONDS = 'zeroshot random_fewshot knn_fewshot afsp_margin afsp_full'
!python manage.py bootstrap --metric comet --conditions {CONDS} --split val --adjacent